# 07 · Run records, replay and trace

**Role of this layer.** Every run is saved as `runs/<run_id>/manifest.json` plus
one output file per step. The manifest records the request, what the LLM
proposed and what the user changed and approved, the exact data versions (file
hashes), library versions and git commit, the registry version, every step's
output hash, the narrative and its grounding check, and stage timings.

* **replay** re-runs the final plan with the recorded approvals and *no LLM*,
  after checking the data files still match, and compares every output hash.
* **trace** answers "where does this number come from?": it walks the plan
  backwards from a step to the source files.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.lineage import load_run, replay, trace
from quantlab.scenarios import SCENARIOS_BY_ID

scenario = SCENARIOS_BY_ID["volatility_flag_success"]  # scenario 1
session = run_request(scenario.request, list(scenario.dataset_ids), "mock", approve=True)
record = load_run(session.run_id, settings.runs_dir)
print("run:", record.run_id, "| status:", record.status)
print("manifest (in the runs folder):", session.manifest_path.relative_to(settings.runs_dir))
print("registry version:", record.registry_version[:16])
print("environment:", record.environment["python"], record.environment["packages"])
pd.DataFrame([d.model_dump() for d in record.datasets])

run: 20261016-175533-ed26b2 | status: completed
manifest (in the runs folder): 20261016-175533-ed26b2/manifest.json
registry version: 7805e316c2b6922f
environment: 3.12.14 {'pandas': '3.0.6', 'numpy': '2.5.4', 'pydantic': '2.14.1', 'openai': '3.31.0', 'anthropic': '1.14.0'}


,dataset_id,fred_id,sha256,retrieved_at,source_url,rows_used,start,end
0,NASDAQCOM,NASDAQCOM,b5fcb00b05eb60874d1484d057a2143ae3beb0387087e1e8f806c88e1412ccb4,2026-10-14T19:24:42Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=NASDAQCOM,14040,1971-02-05,2026-10-13


The saved output file of each step *is* the canonical text that was hashed, so
anyone can verify an output against the manifest with a plain SHA-256:

In [3]:
import hashlib

outputs = settings.runs_dir / record.run_id / "outputs"
for step in record.steps:
    path = next(outputs.glob(f"{step.step_id}_*"))
    same = hashlib.sha256(path.read_bytes()).hexdigest() == step.output_hash
    print(f"{path.name:28} matches manifest: {same}")

s1_load_series.csv           matches manifest: True
s2_resample.csv              matches manifest: True
s3_compute_returns.csv       matches manifest: True
s4_rolling_volatility.csv    matches manifest: True
s5_flag_regime.csv           matches manifest: True


Replay without any LLM:

In [4]:
report = replay(record.run_id, settings)
print(report.message, "| all match:", report.all_match)
pd.DataFrame([s.model_dump() for s in report.steps])

5 of 5 step outputs match; replay status completed, original status completed. | all match: True


,step_id,capability,original_hash,replay_hash,match
0,s1,load_series,3ea1bb2ab2fe50121f5dc3174a3de5cfe07ac6e0343eefc03485f33dc97c282d,3ea1bb2ab2fe50121f5dc3174a3de5cfe07ac6e0343eefc03485f33dc97c282d,True
1,s2,resample,0c44f38c8ce12abc086f1416f5bd5a174afe0cfa3b6b38d52efcfad89d19489f,0c44f38c8ce12abc086f1416f5bd5a174afe0cfa3b6b38d52efcfad89d19489f,True
2,s3,compute_returns,b607a740b7872e1349e5223ab3b22513417c6949a7949ad05bc222e3430d5e04,b607a740b7872e1349e5223ab3b22513417c6949a7949ad05bc222e3430d5e04,True
3,s4,rolling_volatility,35c7d923a74817f15ce486d1019c2b0007f5001931754fb12a0dc64087cdb83d,35c7d923a74817f15ce486d1019c2b0007f5001931754fb12a0dc64087cdb83d,True
4,s5,flag_regime,710fc3814adbd696ac524fdcc762e472597d91906b99eb2d4dd937198bb79256,710fc3814adbd696ac524fdcc762e472597d91906b99eb2d4dd937198bb79256,True


Scenario 6: trace the flags (step s5) back to the NASDAQ file:

In [5]:
lineage = trace(record, "s5")
pd.DataFrame(
    [
        {
            "step": s.step_id,
            "capability": s.capability,
            "inputs": s.inputs,
            "params": s.params,
            "output sha256": s.output_hash[:12],
        }
        for s in lineage.chain
    ]
)

,step,capability,inputs,params,output sha256
0,s5,flag_regime,{'series': 's4'},"{'quantile': 0.9, 'method': 'expanding', 'min_periods': 52}",710fc3814adb
1,s4,rolling_volatility,{'series': 's3'},"{'window': 12, 'annualize': True}",35c7d923a748
2,s3,compute_returns,{'series': 's2'},"{'method': 'simple', 'periods': 1}",b607a740b787
3,s2,resample,{'series': 's1'},"{'freq': 'W', 'how': 'last'}",0c44f38c8ce1
4,s1,load_series,{},{'dataset_id': 'NASDAQCOM'},3ea1bb2ab2fe


In [6]:
pd.DataFrame([s.model_dump() for s in lineage.sources])

,dataset_id,fred_id,sha256,retrieved_at,source_url,rows_used,start,end
0,NASDAQCOM,NASDAQCOM,b5fcb00b05eb60874d1484d057a2143ae3beb0387087e1e8f806c88e1412ccb4,2026-10-14T19:24:42Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=NASDAQCOM,14040,1971-02-05,2026-10-13
